# Notebook 2: Candidate Circuit Discovery and Stability

Colab-ready workflow for discovering candidate circuits in a trained flow-circuits encoder and inspecting their spatial structure and stability.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jacobposchl/model-interpretability.git'
REPO_DIR = Path('/content/model_interpretability' if IN_COLAB else Path.cwd()).resolve()

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/flow_circuits')
    DATA_ROOT  = Path('/content/drive/MyDrive/ctls/data')
else:
    if not (REPO_DIR / 'flow_circuits').exists():
        raise RuntimeError('Run this notebook from the repository root or in Google Colab.')
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'local_notebook_runtime'
    DATA_ROOT  = DRIVE_ROOT / 'data'

BACKBONE_ROOT   = DRIVE_ROOT / 'backbones'
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments'
NOTEBOOK_ROOT   = DRIVE_ROOT / 'notebook_runs'
for path in (DRIVE_ROOT, DATA_ROOT, BACKBONE_ROOT, EXPERIMENT_ROOT, NOTEBOOK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_ROOT = REPO_DIR / 'configs' / 'flow'
print(f"Repo      : {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Data      : {DATA_ROOT}")

In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.discovery import CandidateCircuitDiscoverer
from flow_circuits.training import collect_model_outputs, load_components_from_checkpoint

## Step 1 ? Configure Your Comparison Run

Edit the cell below. This notebook never retrains the flow model.

---

**`TRAINING_MODE`** ? controls how much discovery work runs:

| Mode | Discovery scope | Use for |
|------|------------------|---------|
| `'smoke'` | very small subset | quick pipeline check |
| `'debug'` | medium subset | faster qualitative comparison |
| `'full'` | config defaults | full comparison |

---

**`CONFIG_NAME`** ? should point to an aligned experiment that already produced both `phase_b.pt` and `phase_c.pt`.

---

**Path overrides** ? set these only if your checkpoints or comparison artifacts live outside the canonical experiment directory.


In [ ]:
# ?? Comparison mode ???????????????????????????????????????????????????????????
#   'smoke' = very small discovery subset (fast pipeline check)
#   'debug' = medium discovery subset (faster qualitative comparison)
#   'full'  = full discovery settings from the config
TRAINING_MODE = 'full'  # <- change me

# ?? Model configuration ???????????????????????????????????????????????????????
#   Use an aligned config that already has both phase_b.pt and phase_c.pt
CONFIG_NAME = 'resnet18_aligned'  # <- change me

# ?? Optional path overrides ???????????????????????????????????????????????????
CHECKPOINT_PATH = None  # directory containing phase_b.pt and phase_c.pt
CIRCUITS_PATH   = None  # directory for candidate_circuits_phase_b.json / phase_c.json

# ?? Output directory ??????????????????????????????????????????????????????????
OUTPUT_DIR = str(NOTEBOOK_ROOT / 'nb02')


In [ ]:
import copy
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

_MODE_SETTINGS = {
    'smoke': {
        'label': 'Smoke comparison ? tiny discovery subset',
        'disc_images': 256,
        'disc_bootstrap': 4,
    },
    'debug': {
        'label': 'Debug comparison ? medium discovery subset',
        'disc_images': 2000,
        'disc_bootstrap': 10,
    },
    'full': {
        'label': 'Full comparison ? config discovery settings',
        'disc_images': None,
        'disc_bootstrap': None,
    },
}
_ms = _MODE_SETTINGS[TRAINING_MODE]
print(f"Comparison mode : {TRAINING_MODE}")
print(f"  {_ms['label']}")

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLI_MODULES = {
    'flow-discover': 'flow_circuits.cli.discover',
}

def run_flow_cli(command: str, *args: str) -> None:
    def _stream(cmd):
        import os
        env = os.environ.copy()
        env['PYTHONUNBUFFERED'] = '1'
        process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, cwd=REPO_DIR, env=env,
        )
        while True:
            line = process.stdout.readline()
            if not line:
                break
            print(line, end='', flush=True)
        process.wait()
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)
    try:
        _stream([command, *args])
    except FileNotFoundError:
        _stream([sys.executable, '-m', CLI_MODULES[command], *args])

with open(str(CONFIG_ROOT / f'{CONFIG_NAME}.yaml'), 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

if config['experiment'].get('mode') != 'aligned':
    raise ValueError('Notebook 2 compares phase_b.pt and phase_c.pt, so use an aligned config.')

config['data']['data_dir'] = str(DATA_ROOT)
config['data']['download'] = False
config['logging']['checkpoint_dir'] = str(EXPERIMENT_ROOT / config['experiment']['name'])

def _normalize_dir(value: str | None, expected_suffix: str) -> Path | None:
    if not value:
        return None
    path = Path(value)
    return path.parent if path.suffix == expected_suffix else path

checkpoint_root = _normalize_dir(CHECKPOINT_PATH, '.pt') or Path(config['logging']['checkpoint_dir'])
artifact_root = _normalize_dir(CIRCUITS_PATH, '.json') or checkpoint_root
artifact_root.mkdir(parents=True, exist_ok=True)

effective_config = copy.deepcopy(config)
if TRAINING_MODE != 'full':
    effective_config['discovery']['max_images'] = _ms['disc_images']
    effective_config['discovery']['bootstrap_iterations'] = _ms['disc_bootstrap']

EFFECTIVE_CONFIG = OUTPUT_DIR / f'{TRAINING_MODE}_config.yaml'
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(effective_config, sort_keys=False), encoding='utf-8')

MODEL_ORDER = [('phase_b', 'Phase B'), ('phase_c', 'Phase C')]
MODEL_SPECS = {
    tag: {
        'tag': tag,
        'label': label,
        'checkpoint': checkpoint_root / f'{tag}.pt',
        'circuits': artifact_root / f'candidate_circuits_{tag}.json',
    }
    for tag, label in MODEL_ORDER
}

print()
print(f"Config         : {EFFECTIVE_CONFIG}")
for tag, label in MODEL_ORDER:
    spec = MODEL_SPECS[tag]
    print(f"{label} checkpoint : {spec['checkpoint']}")
    print(f"{label} circuits    : {spec['circuits']}")


## Step 2 ? Discover Circuits Side By Side

This notebook never retrains the flow model.

For each of the two saved checkpoints:
1. verify the checkpoint exists
2. run discovery if the comparison artifact is missing
3. load both artifacts for side-by-side inspection


In [ ]:
missing_checkpoints = [
    f"{spec['label']} checkpoint -> {spec['checkpoint']}"
    for spec in MODEL_SPECS.values()
    if not spec['checkpoint'].exists()
]
if missing_checkpoints:
    raise FileNotFoundError(
        'Notebook 2 requires pre-trained phase_b.pt and phase_c.pt checkpoints.\n'
        + '\n'.join(missing_checkpoints)
    )

artifacts_by_phase = {}
for tag, label in MODEL_ORDER:
    spec = MODEL_SPECS[tag]
    if not spec['circuits'].exists():
        print(f"{label}: circuit artifact not found ? running discovery...")
        run_flow_cli('flow-discover', '--checkpoint', str(spec['checkpoint']), '--output', str(spec['circuits']))
    else:
        print(f"{label}: circuit artifact found ? skipping discovery.")
        print(f"  {spec['circuits']}")
    artifacts_by_phase[tag] = json.loads(spec['circuits'].read_text(encoding='utf-8'))

print() 
for tag, label in MODEL_ORDER:
    artifact = artifacts_by_phase[tag]
    print(f"{label}: {len(artifact['circuits'])} candidate circuit(s) from {artifact['metadata']['n_images']} discovery images")


## Step 3 ? Compare Discovery Outputs

The cells below compare the retained candidate circuits from `phase_b.pt` and `phase_c.pt` side by side.


In [ ]:
import numpy as np

print(f"{'Model':<8} {'Images':>7} {'Circuits':>8} {'Stable':>7} {'Mean nodes':>10} {'Mean purity':>11}")
print(f"{'-'*8} {'-'*7} {'-'*8} {'-'*7} {'-'*10} {'-'*11}")
for tag, label in MODEL_ORDER:
    artifact = artifacts_by_phase[tag]
    circuits = artifact['circuits']
    stable = sum(1 for row in artifact.get('stability_summary', {}).get('per_circuit', []) if row.get('stable'))
    node_counts = [len(c['active_nodes']) for c in circuits]
    purities = [float(c['purity']) for c in circuits if c.get('purity') is not None]
    mean_nodes = np.mean(node_counts) if node_counts else float('nan')
    mean_purity = np.mean(purities) if purities else float('nan')
    print(f"{label:<8} {artifact['metadata']['n_images']:>7} {len(circuits):>8} {stable:>7} {mean_nodes:>10.1f} {mean_purity:>11.3f}")


In [ ]:
import numpy as np

max_plots = max((min(4, len(artifacts_by_phase[tag]['circuits'])) for tag, _ in MODEL_ORDER), default=0)
if max_plots == 0:
    print('No candidate circuits were discovered for either checkpoint.')
else:
    grid_size = next(iter(artifacts_by_phase.values()))['metadata']['grid_size']
    fig, axes = plt.subplots(len(MODEL_ORDER), max_plots, figsize=(4 * max_plots, 3.5 * len(MODEL_ORDER)), squeeze=False)
    for row_idx, (tag, label) in enumerate(MODEL_ORDER):
        circuits = artifacts_by_phase[tag]['circuits'][:max_plots]
        for col_idx in range(max_plots):
            axis = axes[row_idx][col_idx]
            if col_idx >= len(circuits):
                axis.axis('off')
                continue
            circuit = circuits[col_idx]
            heatmap = np.zeros((grid_size, grid_size), dtype=float)
            for _, cell_idx in circuit['active_nodes']:
                row = cell_idx // grid_size
                col = cell_idx % grid_size
                heatmap[row, col] += 1.0
            image = axis.imshow(heatmap, cmap='magma')
            axis.set_title(f"{label} ? C{circuit['id']}\n{len(circuit['active_nodes'])} active nodes")
            plt.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    plt.suptitle('Spatial circuit footprints: Phase B vs Phase C', y=1.02)
    plt.tight_layout()


In [ ]:
import numpy as np

for tag, label in MODEL_ORDER:
    artifact = artifacts_by_phase[tag]
    all_nodes = {tuple(node) for circuit in artifact['circuits'] for node in circuit['active_nodes']}
    purities = [float(c['purity']) for c in artifact['circuits'] if c.get('purity') is not None]
    engagement_sizes = [len(c['engagement_profile']) for c in artifact['circuits']]
    n_total_nodes = artifact['metadata']['n_layers'] * artifact['metadata']['n_cells']
    coverage = len(all_nodes) / max(1, n_total_nodes)
    print(label)
    print(f"  Active node coverage : {len(all_nodes)}/{n_total_nodes} ({coverage:.1%})")
    if purities:
        print(f"  Mean purity          : {np.mean(purities):.3f}")
    if engagement_sizes:
        print(f"  Mean engaged nodes   : {np.mean(engagement_sizes):.1f}")
    print()


In [ ]:
for tag, label in MODEL_ORDER:
    artifact = artifacts_by_phase[tag]
    stable_rows = artifact.get('stability_summary', {}).get('per_circuit', [])
    print(label)
    if not artifact['circuits']:
        print('  No candidate circuits retained.')
        print()
        continue
    print(f"  Top circuits by size:")
    for circuit in sorted(artifact['circuits'], key=lambda row: len(row['active_nodes']), reverse=True)[:5]:
        rep = circuit['representative_node']
        print(
            f"    C{circuit['id']}: {len(circuit['image_set'])} images, {len(circuit['active_nodes'])} nodes, "
            f"rep=(layer {rep[0]}, cell {rep[1]})"
        )
    if stable_rows:
        stable = sum(1 for row in stable_rows if row.get('stable'))
        print(f"  Stable circuits     : {stable}/{len(stable_rows)}")
    print()
